In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import json
import os
import time
import copy
import random
from tqdm import tqdm

# ==========================================
# 1. CONFIG & UTILS
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def get_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    indices = [i for i, label in enumerate(dataset.targets) if label in classes]
    return Subset(dataset, indices)

# --- SURGICAL EXPANSION UTILS ---
def expand_classifier(old_layer, new_num_classes):
    old_out, in_features = old_layer.weight.shape
    new_layer = nn.Linear(in_features, new_num_classes)
    with torch.no_grad():
        new_layer.weight[:old_out] = old_layer.weight
        new_layer.bias[:old_out] = old_layer.bias
    return new_layer

def expand_conv_layer(old_layer, new_in_channels):
    out_ch, old_in_ch, k, _ = old_layer.weight.shape
    new_layer = nn.Conv2d(new_in_channels, out_ch, kernel_size=k, stride=old_layer.stride, padding=old_layer.padding)
    with torch.no_grad():
        new_layer.weight[:, :old_in_ch, :, :] = old_layer.weight
        new_layer.bias[:] = old_layer.bias
        nn.init.kaiming_normal_(new_layer.weight[:, old_in_ch:, :, :], nonlinearity='relu')
    return new_layer

def expand_channel_gate(old_gate, num_old, num_new, ch_per_spec):
    old_in = num_old * ch_per_spec
    new_in = (num_old + num_new) * ch_per_spec
    old_hidden = old_in // 4
    new_hidden = new_in // 4
    
    old_conv1, old_conv2 = old_gate[1], old_gate[3]
    
    new_conv1 = nn.Conv2d(new_in, new_hidden, 1)
    new_conv2 = nn.Conv2d(new_hidden, new_in, 1)
    
    with torch.no_grad():
        new_conv1.weight[:old_hidden, :old_in] = old_conv1.weight
        new_conv1.bias[:old_hidden] = old_conv1.bias
        nn.init.kaiming_normal_(new_conv1.weight[old_hidden:, :], nonlinearity='relu')
        nn.init.kaiming_normal_(new_conv1.weight[:, old_in:], nonlinearity='relu')
        
        new_conv2.weight[:old_in, :old_hidden] = old_conv2.weight
        new_conv2.bias[:old_in] = old_conv2.bias
        nn.init.kaiming_normal_(new_conv2.weight[old_in:, :], nonlinearity='relu')
        nn.init.kaiming_normal_(new_conv2.weight[:, old_hidden:], nonlinearity='relu')
        
    return nn.Sequential(nn.AdaptiveAvgPool2d(1), new_conv1, nn.ReLU(), new_conv2, nn.Sigmoid())

# --- DISTILLATION UTILS ---
def distill_loss(student_logits, teacher_logits, T=2.0):
    p = F.log_softmax(student_logits / T, dim=1)
    q = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(p, q, reduction='batchmean') * (T * T)

def feature_distill_loss(student_proj, teacher_proj):
    loss = 0
    num_shared = len(teacher_proj)
    for i in range(num_shared):
        loss += F.mse_loss(student_proj[i], teacher_proj[i])
    return loss

def anchor_loss(student_logits, teacher_logits):
    return F.mse_loss(student_logits, teacher_logits)

def build_replay_buffer(dataset, classes, per_class=100):
    indices = []
    counts = {k:0 for k in classes}
    for i, label in enumerate(dataset.targets):
        if label in classes and counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
    return Subset(dataset, indices)

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(),
            ResidualBlock(32, 64, 2), 
            ResidualBlock(64, 128, 2), 
            ResidualBlock(128, 64, 1),
            nn.AdaptiveAvgPool2d((8, 8))
        )
    def forward(self, x): return self.net(x)

class Broker(nn.Module):
    def __init__(self, n_specs, ch, n_classes):
        super().__init__()
        self.projections = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(n_specs)])
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(n_specs*ch, n_specs*ch//4, 1), nn.ReLU(), 
            nn.Conv2d(n_specs*ch//4, n_specs*ch, 1), nn.Sigmoid()
        )
        self.spatial_gate = nn.Sequential(nn.Conv2d(2, 1, 7, 1, 3), nn.Sigmoid())
        self.bottleneck = nn.Sequential(nn.Conv2d(n_specs*ch, ch, 1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(ch, n_classes)

    def forward(self, feats):
        proj = [p(f) for p, f in zip(self.projections, feats)]
        z_raw = torch.cat(proj, 1)
        w_c = self.channel_gate(z_raw)
        z = z_raw * w_c
        avg_out = torch.mean(z, 1, keepdim=True)
        max_out, _ = torch.max(z, 1, keepdim=True)
        w_s = self.spatial_gate(torch.cat([avg_out, max_out], 1))
        z = z * w_s
        out = self.bottleneck(z)
        return self.classifier(self.pool(out).flatten(1)), proj

class KFN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist(), Specialist()])
        self.broker = Broker(2, 64, n_classes)
    def forward(self, x): return self.broker([s(x) for s in self.specialists])

def cka_loss(feats):
    loss = 0
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            x, y = feats[i].flatten(1), feats[j].flatten(1)
            loss += torch.sum((x @ x.t()) * (y @ y.t())) / (x.shape[0]**2)
    return loss / (len(feats)*(len(feats)-1)/2 + 1e-8)

# ==========================================
# 3. EXPERIMENT EXECUTION (SOTA POLISH)
# ==========================================
def run():
    set_deterministic(42)
    device = get_device()
    os.makedirs("logs_sota", exist_ok=True)
    
    # Correct Stats
    stats = ((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
    train_T = transforms.Compose([
        transforms.RandomCrop(32, 4), transforms.RandomHorizontalFlip(), 
        transforms.ToTensor(), transforms.Normalize(*stats)
    ])
    test_T = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    
    full_train = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_T)
    full_test = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_T)
    
    loader_A = DataLoader(get_class_subset(full_train, [0,1,2,3,4]), 128, shuffle=True, num_workers=2)
    loader_B = DataLoader(get_class_subset(full_train, [5,6,7,8,9]), 128, shuffle=True, num_workers=2)
    test_A = DataLoader(get_class_subset(full_test, [0,1,2,3,4]), 128, num_workers=2)
    test_B = DataLoader(get_class_subset(full_test, [5,6,7,8,9]), 128, num_workers=2)

    # --- PHASE 1: Strong Base ---
    print("Phase 1: Training Task A (60 Epochs)...")
    model = KFN(5).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
    
    for ep in range(60): 
        model.train()
        for x, y in tqdm(loader_A, desc=f"Base Ep {ep+1}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits, feats = model(x)
            loss = F.cross_entropy(logits, y) + 0.1*cka_loss(feats)
            loss.backward()
            opt.step()
        sched.step()
            
    acc_A = evaluate(model, test_A, device)
    print(f"Task A Base Acc: {acc_A:.2f}%")
    
    # --- PREPARE EXPANSION ---
    print("Preparing Expansion...")
    teacher = copy.deepcopy(model)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    
    replay_data = build_replay_buffer(full_train, [0,1,2,3,4], per_class=100)
    replay_loader = DataLoader(replay_data, 32, shuffle=True)
    replay_iter = iter(replay_loader)
    
    # Pre-train Independent Specialist
    print("Pre-training New Specialist...")
    new_spec = Specialist().to(device)
    temp_head = nn.Linear(64, 5).to(device)
    spec_opt = torch.optim.AdamW(list(new_spec.parameters()) + list(temp_head.parameters()), lr=1e-3)
    for ep in range(20):
        new_spec.train()
        for x, y in tqdm(loader_B, desc=f"Spec Ep {ep+1}", leave=False):
            x, y = x.to(device), (y - 5).to(device)
            spec_opt.zero_grad()
            feats = new_spec(x)
            pooled = F.adaptive_avg_pool2d(feats, 1).flatten(1)
            loss = F.cross_entropy(temp_head(pooled), y)
            loss.backward()
            spec_opt.step()
            
    # Surgical Integration
    print("Integrating...")
    model.specialists.append(new_spec)
    old_b = model.broker
    new_b = Broker(3, 64, 10).to(device)
    
    new_b.projections[0].load_state_dict(old_b.projections[0].state_dict())
    new_b.projections[1].load_state_dict(old_b.projections[1].state_dict())
    new_b.spatial_gate.load_state_dict(old_b.spatial_gate.state_dict())
    new_b.classifier = expand_classifier(old_b.classifier, 10).to(device)
    new_b.bottleneck[0] = expand_conv_layer(old_b.bottleneck[0], 192).to(device)
    new_b.channel_gate = expand_channel_gate(old_b.channel_gate, 2, 1, 64).to(device)
    
    model.broker = new_b
    
    for p in model.specialists[0].parameters(): p.requires_grad = False
    for p in model.specialists[1].parameters(): p.requires_grad = False
    model.specialists[0].eval()
    model.specialists[1].eval()
    
    # --- PHASE 2: SOTA HYBRID TRAINING ---
    print("Phase 2: SOTA Hybrid Training...")
    
    # Differential LR
    param_groups = [
        {'params': model.specialists[2].parameters(), 'lr': 5e-4}, 
        {'params': model.broker.parameters(), 'lr': 1e-4}
    ]
    opt = torch.optim.AdamW(param_groups)
    
    # Tuned Weights
    MAX_LOGIT = 2.0
    MAX_FEAT = 2.5
    MAX_REPLAY = 1.0
    MAX_ANCHOR = 2.0 
    
    TOTAL_EPS = 25
    
    for ep in range(TOTAL_EPS):
        model.train()
        model.specialists[0].eval()
        model.specialists[1].eval()
        
        # --- CURRICULUM + ANNEALING ---
        if ep < 5:
            stage = 0; curr_anchor=0; curr_feat=0; curr_replay=0.1
            curr_T = 2.0 
            for p in model.broker.parameters(): p.requires_grad = False
        elif ep < 10:
            stage = 1; curr_anchor=0; curr_feat=0; curr_replay=0.2
            curr_T = 2.0
            for p in model.broker.parameters(): p.requires_grad = True
            opt.param_groups[0]['lr'] = 1e-3 
        elif ep < 20:
            stage = 2
            curr_T = 2.0
            opt.param_groups[0]['lr'] = 5e-4 
            progress = (ep - 10) / 10.0
            curr_anchor = MAX_ANCHOR * progress
            curr_feat = MAX_FEAT * progress
            curr_replay = 0.2 + (MAX_REPLAY - 0.2) * progress
        else:
            stage = 3 # Lockdown
            # FIX 2 & 3: Anchor Decay + Temp Annealing
            curr_anchor = 1.5 # Relaxed from 2.0
            curr_T = 3.0      # Smoother gradients
            curr_feat = MAX_FEAT
            curr_replay = MAX_REPLAY

        desc = f"Ep {ep+1} | A:{curr_anchor:.1f} T:{curr_T:.1f}"
        
        for x_B, y_B in tqdm(loader_B, desc=desc, leave=False):
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            do_replay = random.random() < curr_replay
            if do_replay or stage > 1: 
                try: x_A, y_A = next(replay_iter)
                except: 
                    replay_iter = iter(replay_loader)
                    x_A, y_A = next(replay_iter)
                x_A, y_A = x_A.to(device), y_A.to(device)
            
            opt.zero_grad()
            
            logits_B, feats_B = model(x_B)
            
            if stage == 0:
                 loss = F.cross_entropy(logits_B, y_B)
            else:
                 loss_B = F.cross_entropy(logits_B, y_B)
                 
                 loss_replay, loss_logit, loss_feat, loss_anchor = 0,0,0,0
                 
                 if do_replay or stage > 1:
                     logits_A_student, proj_A_student = model(x_A)
                     with torch.no_grad():
                         logits_A_teacher, proj_A_teacher = teacher(x_A)
                     
                     loss_replay = F.cross_entropy(logits_A_student, y_A)
                     # FIX 3: Dynamic Temperature T
                     loss_logit = distill_loss(logits_A_student[:, :5], logits_A_teacher[:, :5], T=curr_T)
                     loss_anchor = anchor_loss(logits_A_student[:, :5], logits_A_teacher[:, :5])
                     loss_feat = feature_distill_loss(proj_A_student, proj_A_teacher)
                 
                 loss_cka = cka_loss(feats_B)
                 
                 loss = (loss_B + 
                         curr_replay * loss_replay + 
                         curr_anchor * loss_anchor + 
                         curr_feat * loss_feat + 
                         0.05 * loss_cka)
            
            loss.backward()
            
            # --- FIX 1: CLEAR MOMENTUM BUFFERS ---
            if stage > 0:
                with torch.no_grad():
                    # 1. Hard Reset Weights
                    model.broker.classifier.weight[:5, :] = teacher.broker.classifier.weight[:5, :]
                    model.broker.classifier.bias[:5] = teacher.broker.classifier.bias[:5]
                    
                    # 2. Hard Reset Momentum (The SOTA Fix)
                    for param in [model.broker.classifier.weight, model.broker.classifier.bias]:
                        if param in opt.state:
                            state = opt.state[param]
                            if "exp_avg" in state:
                                # For weights: shape is [10, 64], so [:5] works
                                # For bias: shape is [10], so [:5] works
                                state["exp_avg"][:5] = 0.0
                                state["exp_avg_sq"][:5] = 0.0
            
            opt.step()
            
    # --- FINAL VERDICT ---
    acc_A_final = evaluate(model, test_A, device)
    acc_B_final = evaluate(model, test_B, device)
    
    print("\nFINAL RESULTS (SOTA Polished):")
    print(f"Task A (Old): {acc_A:.2f}% -> {acc_A_final:.2f}%")
    print(f"Task B (New): {acc_B_final:.2f}%")
    print(f"Retention: {acc_A_final/acc_A:.1%}")

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

if __name__ == "__main__":
    run()

100%|██████████| 170M/170M [00:25<00:00, 6.65MB/s] 


Phase 1: Training Task A (60 Epochs)...


Task A Base Acc: 72.54%
Preparing Expansion...
Pre-training New Specialist...


Integrating...
Phase 2: SOTA Hybrid Training...



FINAL RESULTS (SOTA Polished):
Task A (Old): 72.54% -> 51.84%
Task B (New): 68.96%
Retention: 71.5%
